In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Create a permanent assets folder in Google Drive.
DRIVE_ASSETS = '/content/drive/MyDrive/hybrid_vtg_assets'
os.makedirs(DRIVE_ASSETS, exist_ok=True)
print(f'Assets will be permanently saved to: {DRIVE_ASSETS}')


In [ ]:
import os

REPO_NAME = 'hybrid_vtg'
GITHUB_URL = 'https://github.com/TienDat8605/TemporalGroundings.git'
BRANCH_NAME = 'BGS-Scrape'

if not os.path.exists(REPO_NAME) and not os.path.exists('src/hybrid_vtg'):
    !git clone -b {BRANCH_NAME} {GITHUB_URL} {REPO_NAME}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}
    !git fetch origin
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

%cd hybrid_vtg
print('Current working directory:', os.getcwd())
if not os.path.exists('./assets'):
    os.symlink(DRIVE_ASSETS, './assets')
    print('Symlinked ./assets -> Google Drive.')


In [ ]:
BRANCH_NAME = 'Boundary-based'
!git fetch origin
!git checkout {BRANCH_NAME}
!git pull origin {BRANCH_NAME}


In [ ]:
# Install dependencies cleanly into the Colab environment.
%pip install -q 'opencv-python-headless>=4.9,<5' 'numpy>=1.26,<2.1'
%pip install -e '.[downloads,test]'

import sys
if os.path.abspath('src') not in sys.path:
    sys.path.insert(0, os.path.abspath('src'))


In [ ]:
# Persist Hugging Face downloads and optional precomputed SigLIP2 scout assets.
os.environ['HF_HOME'] = f'{DRIVE_ASSETS}/huggingface'
SCOUT_FEATURE_ROOT = f'{DRIVE_ASSETS}/features/scouts'
os.makedirs(SCOUT_FEATURE_ROOT, exist_ok=True)


In [ ]:
# Run once only when OMTG is not already available in ./assets/datasets/omtg.
#!PYTHONPATH=src python -m hybrid_vtg.cli download omtg --root ./assets --accept-licenses


## ASGDE-OMTG Smoke Run

`asgde-omtg` uses frozen SigLIP2 at 1 FPS for full-video retrieval and one frozen Qwen3-VL-4B multi-span grounding call. It requires dense evidence; do not enable Mage or SemVID pruning. The first run builds SigLIP2 cache artifacts under the results cache, while any compatible Drive feature artifacts passed through `--feature-root` are reused.


In [ ]:
# Keep this as a small smoke run. Increase --subset only after inspecting telemetry below.
!PYTHONPATH=src python -m hybrid_vtg.cli run \
  --benchmark omtg \
  --data ./assets/datasets/omtg \
  --model qwen3-vl-4b \
  --method asgde-omtg \
  --subset 5 \
  --seed 42 \
  --feature-root {SCOUT_FEATURE_ROOT}


In [ ]:
from collections import Counter
from pathlib import Path
import json

ROOT = Path('/content/hybrid_vtg/hybrid_vtg')
RUN = ROOT / 'results/runs/omtg/qwen3-vl-4b/asgde-omtg/seed-42'

def load_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines()]

rows = load_jsonl(RUN / 'predictions.jsonl')
errors = load_jsonl(RUN / 'errors.jsonl')
telemetry = [row['prediction']['telemetry'] for row in rows]

print('successful predictions:', len(rows))
print('errors:', len(errors))
print('route modes:', Counter(value['route_mode'] for value in telemetry))
print('frame budgets:', Counter(value['frame_budget'] for value in telemetry))
print('peak counts:', Counter(value['peak_count'] for value in telemetry))
print('SigLIP2 cache hits:', Counter(value['scout_cached'] for value in telemetry))

for row in rows:
    prediction = row['prediction']
    details = prediction['telemetry']
    print(f"\nID: {row['id']}")
    print('query:', row['query'])
    print('targets:', row['targets'])
    print('spans:', prediction['spans'])
    print('route:', details['route_mode'], 'budget:', details['frame_budget'])
    print('corridors:', details['selected_corridors'])
    print('role counts:', details['observation_role_counts'])
    assert details['total_frames'] == details['frame_budget']
    assert details['encoder_calls'] == details['primary_grounder_calls'] == 1


## 5. View Evaluation Results

Display generated `RESULTS.md` directly in the notebook.


In [ ]:
from IPython.display import Markdown, display

results_path = 'results/RESULTS.md'
if os.path.exists(results_path):
    with open(results_path) as handle:
        display(Markdown(handle.read()))
else:
    print('RESULTS.md not found. Make sure a benchmark run has completed.')
